# Week 9: Confusion Matrix & ROC Curves — Beyond Simple Accuracy
**TMI-102 | PaAC, IIT Roorkee**

---

### What you'll do this week
1. Plot a **confusion matrix** (which classes get mixed up?)
2. Plot **ROC curves** with AUC scores
3. Get **precision, recall, F1** per class
4. Compare all 3 models side-by-side

### Why accuracy isn't enough
"94% accurate" doesn't tell you *which* galaxies it gets wrong. Maybe it's perfect at spirals but terrible at mergers.

Note: this version uses the **3-class dataset** (elliptical, merger, spiral), so the confusion matrices are 3×3 and ROC is **one-vs-rest** (3 curves per model). All three retrained models are used: Week 4 frozen, Week 5 full fine-tune, Week 6 LoRA. The test set is *not* balanced (mergers are rarer), so besides accuracy I also report balanced accuracy and macro-F1. Class order is the one Keras uses (alphabetical): elliptical, merger, spiral.

**Setup for this notebook:** add (1) the Galaxy Zoo competition data and (2) the dataset holding the three `.keras` models as inputs. The next cell rebuilds `Galaxy_dataset_3class` with the same recipe and seed as the retrain notebook, so the test images are the same ones the models were evaluated on.

In [ ]:
# ── Build Galaxy_dataset_3class from the competition files (runs once; skipped if the folder already exists) ──
import os, zipfile
import numpy as np, pandas as pd

COMP = "/kaggle/input/competitions/galaxy-zoo-the-galaxy-challenge"
BUILD_DIR = "/kaggle/working/Galaxy_dataset_3class"
ELL_T, SPI_T, MERGER_T, SEED = 0.80, 0.80, 0.30, 42     # merger = Class8.6 >= 0.30 (change here if mentors say otherwise)

if not os.path.isdir(BUILD_DIR + "/train"):
    from sklearn.model_selection import train_test_split
    df = pd.read_csv(f"{COMP}/training_solutions_rev1.zip", compression="zip")

    # --- elliptical + spiral: EXACTLY the Week 3 recipe (same thresholds, same seed, same split) ---
    df["class"] = np.select([df["Class1.1"] >= ELL_T, df["Class1.2"] >= SPI_T], ["elliptical", "spiral"], default="Tie")
    filtered = df[df["class"].isin(["elliptical", "spiral"])].copy()
    min_cnt = min(filtered["class"].value_counts())
    balanced = pd.concat([
        filtered[filtered["class"] == "spiral"].sample(n=min_cnt, random_state=SEED),
        filtered[filtered["class"] == "elliptical"].sample(n=min_cnt, random_state=SEED)
    ]).sample(frac=1.0, random_state=SEED).reset_index(drop=True)
    tr, tmp = train_test_split(balanced, test_size=0.30, random_state=SEED, stratify=balanced["class"])
    va, te = train_test_split(tmp, test_size=0.50, random_state=SEED, stratify=tmp["class"])

    # --- merger: simply ADDED as a third class. Galaxies already used as elliptical/spiral are skipped,
    #     so one image never has two labels and the two old classes stay exactly as they were. ---
    mer_all = df[df["Class8.6"] >= MERGER_T]
    mer = mer_all[~mer_all["GalaxyID"].isin(balanced["GalaxyID"])].copy()
    mer["class"] = "merger"
    print(f"merger candidates {len(mer_all)} -> {len(mer)} after skipping {len(mer_all) - len(mer)} already used as elliptical/spiral")
    mtr, mtmp = train_test_split(mer, test_size=0.30, random_state=SEED)
    mva, mte = train_test_split(mtmp, test_size=0.50, random_state=SEED)

    splits = [(tr, "train"), (va, "val"), (te, "test"), (mtr, "train"), (mva, "val"), (mte, "test")]
    for s in ["train", "val", "test"]:
        for c in ["elliptical", "merger", "spiral"]:
            os.makedirs(f"{BUILD_DIR}/{s}/{c}", exist_ok=True)
    target, rows = {}, []
    for d, s in splits:
        for gid, c in zip(d["GalaxyID"], d["class"]):
            target[f"{gid}.jpg"] = (s, c); rows.append((gid, c, s))

    n_ext = 0
    with zipfile.ZipFile(f"{COMP}/images_training_rev1.zip") as z:
        for member in z.namelist():
            base = os.path.basename(member)
            if base in target:
                s, c = target[base]
                with open(f"{BUILD_DIR}/{s}/{c}/{base}", "wb") as f:
                    f.write(z.read(member))
                n_ext += 1
    man = pd.DataFrame(rows, columns=["GalaxyID", "cls", "split"])
    man.to_csv(f"{BUILD_DIR}/manifest.csv", index=False)
    print("extracted", n_ext, "of", len(man))
    print(man.groupby(["split", "cls"]).size().unstack())
    print("Check vs Week 3: elliptical/spiral should be 5692 / 1220 / 1220 each.")
else:
    print("Dataset already built at", BUILD_DIR)

In [ ]:
import glob, os, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import tensorflow as tf
from tensorflow.keras import layers
from sklearn.metrics import (confusion_matrix, classification_report, roc_curve, auc,
                             f1_score, balanced_accuracy_score)
from sklearn.preprocessing import label_binarize
from tensorflow.keras.applications.resnet50 import preprocess_input
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
@tf.keras.utils.register_keras_serializable(package='tmi102')
class LoRADense(layers.Layer):
    """Dense layer with a LoRA adapter: output = x@W + b + (x@A@B) * (alpha/rank).  W, b frozen; A, B trainable."""
    def __init__(self, units, rank=4, alpha=32, activation=None, **kwargs):
        super().__init__(**kwargs)
        self.units, self.rank, self.alpha = units, rank, alpha
        self.scaling = alpha / rank
        self.activation_fn = tf.keras.activations.get(activation)

    def build(self, input_shape):
        input_dim = input_shape[-1]
        self.W = self.add_weight(name='kernel', shape=(input_dim, self.units), initializer='glorot_uniform', trainable=False)
        self.b = self.add_weight(name='bias', shape=(self.units,), initializer='zeros', trainable=False)
        self.A = self.add_weight(name='lora_A', shape=(input_dim, self.rank), initializer='glorot_normal', trainable=True)
        self.B = self.add_weight(name='lora_B', shape=(self.rank, self.units), initializer='zeros', trainable=True)

    def call(self, x):
        h = tf.matmul(x, self.W) + self.b
        h = h + tf.matmul(tf.matmul(x, self.A), self.B) * self.scaling
        return self.activation_fn(h) if self.activation_fn else h

    def get_config(self):
        cfg = super().get_config()
        cfg.update(units=self.units, rank=self.rank, alpha=self.alpha,
                   activation=tf.keras.activations.serialize(self.activation_fn))
        return cfg

In [ ]:
def find(name):
    return glob.glob(f'/kaggle/input/**/{name}', recursive=True)[0]

MODEL_FILES = {'Week 4 Frozen': 'w4_frozen_3class.keras',
               'Week 5 Full FT': 'w5_full_3class.keras',
               'Week 6 LoRA (r=4)': 'w6_lora_3class.keras'}
models = {k: tf.keras.models.load_model(find(v), custom_objects={'LoRADense': LoRADense}) for k, v in MODEL_FILES.items()}
MAIN = 'Week 5 Full FT'

DATA = BUILD_DIR   # built by the cell above
test_gen = ImageDataGenerator(preprocessing_function=preprocess_input).flow_from_directory(
    f'{DATA}/test', target_size=(224, 224), batch_size=32, class_mode='categorical', shuffle=False)
CLASS_NAMES = [c for c, _ in sorted(test_gen.class_indices.items(), key=lambda kv: kv[1])]
print(test_gen.class_indices)

y_true = test_gen.classes
probs, preds = {}, {}
for k, m in models.items():
    test_gen.reset()
    probs[k] = m.predict(test_gen, verbose=0)
    preds[k] = probs[k].argmax(1)
    print(f"{k:<20} accuracy {np.mean(preds[k] == y_true) * 100:.2f}%   probs shape {probs[k].shape}")
print("Test images per class:", dict(zip(CLASS_NAMES, np.bincount(y_true).tolist())))

### Task 1: Confusion Matrix (15 pts)
First the main model (Week 5) with raw counts, then all three side by side as row-percentages (each row sums to 100% so the smaller merger class is readable).

In [ ]:
def plot_cm(y_true, y_pred, title='Confusion Matrix'):
    cm = confusion_matrix(y_true, y_pred)
    plt.figure(figsize=(6, 5))
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
    plt.title(title, fontweight='bold', fontsize=13)
    plt.ylabel('True Label'); plt.xlabel('Predicted Label')
    plt.tight_layout(); plt.show()
    print(f"Correct predictions: {np.trace(cm)}/{cm.sum()} ({np.trace(cm) / cm.sum() * 100:.1f}%)")
    off = cm.copy(); np.fill_diagonal(off, 0)
    i, j = np.unravel_index(off.argmax(), off.shape)
    print(f"Largest single confusion: true {CLASS_NAMES[i]} → predicted {CLASS_NAMES[j]}: {off[i, j]} images "
          f"({off[i, j] / cm[i].sum() * 100:.1f}% of all true {CLASS_NAMES[i]})")
    return cm

cm_main = plot_cm(y_true, preds[MAIN], f'{MAIN} — Confusion Matrix (counts)')

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8))
for ax, (k, p) in zip(axes, preds.items()):
    cm = confusion_matrix(y_true, p)
    sns.heatmap(cm / cm.sum(1, keepdims=True) * 100, annot=True, fmt='.1f', cmap='Blues', vmin=0, vmax=100,
                xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES, ax=ax, cbar=False)
    ax.set_title(k); ax.set_xlabel('Predicted'); ax.set_ylabel('True')
plt.suptitle('Row-normalised confusion matrices (% of each true class)', fontweight='bold')
plt.tight_layout(); plt.show()

### Task 2: Classification Report (10 pts)

In [ ]:
for k, p in preds.items():
    print(f"===== {k} =====")
    print(classification_report(y_true, p, target_names=CLASS_NAMES, digits=3))

### Task 3: ROC Curves (15 pts)
One-vs-rest: for each class, "this class" vs "everything else".

In [ ]:
y_bin = label_binarize(y_true, classes=list(range(len(CLASS_NAMES))))
colors = ['#00e5ff', '#f472b6', '#fbbf24']

def plot_roc(ax, y_bin, y_probs, title):
    aucs = []
    for i, (cls, col) in enumerate(zip(CLASS_NAMES, colors)):
        fpr, tpr, _ = roc_curve(y_bin[:, i], y_probs[:, i])
        a = auc(fpr, tpr); aucs.append(a)
        ax.plot(fpr, tpr, color=col, lw=2, label=f'{cls} (AUC = {a:.3f})')
    ax.plot([0, 1], [0, 1], 'k--', alpha=0.4)
    ax.set_xlabel('False Positive Rate'); ax.set_ylabel('True Positive Rate')
    ax.set_title(title, fontweight='bold'); ax.legend(loc='lower right', fontsize=9); ax.grid(True, alpha=0.3)
    return aucs

fig, axes = plt.subplots(1, 3, figsize=(17, 5))
aucs = {}
for ax, (k, pr) in zip(axes, probs.items()):
    aucs[k] = plot_roc(ax, y_bin, pr, f'{k} — ROC (one-vs-rest)')
plt.tight_layout(); plt.show()

### Side-by-side comparison of the 3 models

In [ ]:
print(f"{'Model':<20}{'Acc':>8}{'BalAcc':>8}{'MacroF1':>9}{'MacroAUC':>10}   recall: " + "  ".join(f"{c:>10}" for c in CLASS_NAMES))
for k, p in preds.items():
    cm = confusion_matrix(y_true, p); rec = cm.diagonal() / cm.sum(1)
    print(f"{k:<20}{np.mean(p == y_true) * 100:>7.2f}%{balanced_accuracy_score(y_true, p) * 100:>7.2f}%"
          f"{f1_score(y_true, p, average='macro'):>9.4f}{np.mean(aucs[k]):>10.4f}   " + "  ".join(f"{r * 100:>9.1f}%" for r in rec))

### Q1 (5 pts)
Looking at your confusion matrix, which two classes get confused most often? Why does this make sense physically? (Hint: think about what merger galaxies can look like)
*(The "Largest single confusion" line under the Week 5 matrix and the row-normalised plots give you the answer. Check merger **recall** in the comparison table — and remember a merger can be two smooth blobs touching, or a spiral with a distorted arm.)*

> *Write your answer here.*

### Q2 (5 pts)
What does AUC = 0.99 mean in plain English?
*(Your earlier pairwise-ranking explanation still applies — now it's "a random galaxy of this class vs. a random galaxy that is not this class", one-vs-rest. Use your own AUC from the table above.)*

> *Write your answer here.*

---
**Submit as:** `Week_9_YourName.ipynb`